# Chroma 是什么 + 怎么向量化

## 0. 先校准一下你的当前理解

> 你说："chroma 它可以将原本庞大的数据进行向量化成一种固定格式，方便模型引入"

这句话**有一半对**，但混了两件事，需要拆开：

| 你的描述 | 实际负责的组件 |
|---|---|
| "把数据**向量化**" | 这件事**不是 Chroma 做的**，是 **embedding model**（如 `all-MiniLM-L6-v2`、`text-embedding-3-small`、`BGE`、`PubMedBERT` 等）做的 |
| "存成固定格式，方便模型引入" | **这才是 Chroma 的核心职能**：存储 + 建索引 + 相似度检索 |

一句话纠正：

> **Chroma 不向量化，Chroma 只存向量。向量化由 embedding model 负责。**

但实际使用时 Chroma 提供一个便利接口：你给它指定一个 `embedding_function`，然后 `add(documents=[...文本...])` 时它会**自动调那个 embedding model** 把文本转成向量再存。所以"用户视角"看起来像是"丢文本进去就行"，但底层分工要清楚。

---

## 1. Chroma 在 RAG 里的角色

一条标准 RAG 链路：

```
原始文档 ─► 切块(chunking) ─► [embedding model] ─► 向量 ─► [Chroma 存储+索引]
                                                                  │
                                                                  │ 检索
用户问题 ─► [embedding model] ─► 查询向量 ──────────────► top-k 相似 chunk
                                                                  │
                                                                  ▼
                                                         拼进 prompt 喂给 LLM
```

`[embedding model]` 和 `[Chroma]` 是两个独立组件，常被一起谈到所以容易混。

**Chroma 提供的能力**：

| 能力 | 说明 |
|---|---|
| 存储 | 一份「向量 + 原文 + metadata」的三元组库 |
| 索引 | 内部用 HNSW（Hierarchical Navigable Small World）建近邻索引，O(log N) 检索 |
| 相似度检索 | 默认余弦距离 / L2 / 内积，返回 top-k 最相似条目 |
| Metadata 过滤 | 例如 `where={"pub_year": "2024"}` 只在 2024 年文献里检索 |
| 持久化 | `PersistentClient(path=...)` 自动落盘成 sqlite + parquet 文件 |

**Chroma 不做的事**：

- ❌ 不训练 embedding 模型
- ❌ 不做 chunking
- ❌ 不调 LLM
- ❌ 不做精排（rerank）

## 2. "向量化"到底是怎么做的（embedding 模型干的事）

把任意一段文本变成一个**固定维度的浮点向量**，让"语义相近"的文本在向量空间里"距离相近"。

### 2.1 一个直观例子

| 文本 | 向量（举例，384 维只列前 5 维） |
|---|---|
| `"diabetes is a chronic metabolic disease"` | `[ 0.021, -0.117, 0.083, 0.056, -0.044, ...]` |
| `"糖尿病是一种慢性代谢疾病"` | `[ 0.019, -0.121, 0.079, 0.061, -0.040, ...]` |
| `"the dog barked loudly at the mailman"` | `[-0.092, 0.034, -0.011, 0.117, 0.078, ...]` |

前两条虽然语言不同但意思一致，向量**几乎重合**（余弦相似度 ≈ 0.85+）；第三条和它们语义无关，向量方向差得远（余弦相似度 ≈ 0.05 甚至负数）。

### 2.2 模型怎么算出这个向量

以 `sentence-transformers/all-MiniLM-L6-v2`（22 MB，我们最小 RAG 准备用的就是它）为例：

1. **Tokenize**：文本 → token id 序列  
   `"diabetes is a chronic disease"` → `[101, 14924, 2003, 1037, 7517, 4295, 102]`

2. **过 Transformer encoder**：6 层 self-attention，输出每个 token 一个 384 维向量。

3. **Pooling**：把 N 个 token 向量聚合成 1 个 sentence 向量  
   常见做法：mean pooling（对所有 token 向量取平均），或者取 `[CLS]` token 的向量。

4. **归一化**（可选但常做）：把向量长度归到 1，这样后续用点积就等价于余弦相似度。

最终得到固定的 **384 维向量**，无论输入是 5 个词还是 500 个词。

> "embedding 模型"和"LLM"用的是**同一类底层结构**（Transformer），但用法完全不同：
> - LLM：自回归生成下一个 token
> - Embedding：把整段文本"压"成一个固定向量

### 2.3 维度从哪来？

每个 embedding 模型**预先决定好**输出维度：

| 模型 | 维度 | 模型大小 |
|---|---|---|
| `all-MiniLM-L6-v2` | 384 | 22 MB |
| `BAAI/bge-small-en` | 384 | 33 MB |
| `BAAI/bge-base-en` | 768 | 110 MB |
| `pritamdeka/S-PubMedBert-MS-MARCO`（医学专用） | 768 | 420 MB |
| OpenAI `text-embedding-3-small` | 1536 | 闭源 API |

**维度越高 ≠ 一定越好**。常见权衡：维度高检索质量略好，但存储和检索都更慢。RAG 起步用 384 维就够了。

> **同一个 Chroma 库里所有向量必须是同一模型出的**（维度对齐 + 语义空间一致），中途换 embedding 模型必须重新入库。

## 3. Chroma 内部到底存了什么

Chroma 落盘后会在你指定的目录里生成类似这样的结构：

```
chroma_db/
├── chroma.sqlite3              # 主元数据库（集合表、文档表、元数据表）
└── <uuid>/                     # 每个 collection 一个目录
    ├── data_level0.bin         # HNSW 索引主数据（向量本体 + 邻居关系）
    ├── header.bin              # 索引 header（维度、距离函数、参数）
    ├── length.bin              # 每个向量长度
    └── link_lists.bin          # HNSW 的图连接
```

**每条记录的三元组**：

| 字段 | 例子 | 谁产生的 |
|---|---|---|
| `id` | `"PMC8774754_chunk_3"` | 用户给（保证唯一） |
| `document`（原文） | `"Primary ciliary dyskinesia is a rare ..."` | 用户给 |
| `embedding`（向量） | `[0.021, -0.117, ..., 0.044]`（384 维） | embedding 模型给 |
| `metadata`（结构化标签） | `{"pmcid": "PMC8774754", "journal": "Diagnostics", "pub_year": "2022"}` | 用户给 |

> Chroma 把 `document` 原文也存着，所以检索完直接能拿到文本，不用再回原始数据集查表。这是它和 FAISS 的一个差别（FAISS 只存向量，要自己维护 id↔原文的映射）。

---

## 4. 最小可运行示例（不依赖 LangChain，直接看 Chroma 原生 API）

下方代码可以直接在 `med-rag-verify` 环境里跑，**不需要额外装任何包**。

注意：示例**没有显式传 `embedding_function`**，此时 Chroma 自动走它内置的 `DefaultEmbeddingFunction`——基于 ONNX Runtime + 自带的 `all-MiniLM-L6-v2`，首次运行下载约 79 MB 到 `~/.cache/chroma/onnx_models/`，**不依赖 `sentence_transformers`**。

> 想换成 BGE / PubMedBERT 等其他 embedding 模型时，再 `pip install sentence_transformers` 并显式传 `embedding_function=SentenceTransformerEmbeddingFunction(...)`。这一步留到「最小 RAG」阶段做（见 `01 验证模型/schedule.md` 末尾 §6.1）。

In [3]:
import chromadb

client = chromadb.Client()

# 不传 embedding_function -> Chroma 走内置 DefaultEmbeddingFunction（ONNX 版 MiniLM）
# 用 get_or_create_* 是为了重跑这个 cell 时不会报 "collection already exists"
col = client.get_or_create_collection(name="demo_disease")

col.add(
    ids=["d1", "d2", "d3", "d4"],
    documents=[
        "Diabetes is a chronic metabolic disease characterized by high blood glucose.",
        "糖尿病是一种慢性代谢疾病，特征是血糖升高。",
        "Hypertension is a condition where blood pressure is persistently elevated.",
        "The dog barked loudly at the mailman this morning.",
    ],
    metadatas=[
        {"topic": "diabetes", "lang": "en"},
        {"topic": "diabetes", "lang": "zh"},
        {"topic": "hypertension", "lang": "en"},
        {"topic": "irrelevant", "lang": "en"},
    ],
)

print(f"集合中有 {col.count()} 条记录\n")

集合中有 4 条记录



In [4]:
res = col.query(
    query_texts=["what is diabetes?"],
    n_results=3,
)

print("=== query: what is diabetes? ===\n")
for rank, (doc, meta, dist) in enumerate(zip(
        res["documents"][0], res["metadatas"][0], res["distances"][0]), 1):
    print(f"[{rank}] distance={dist:.4f}  meta={meta}")
    print(f"    {doc}\n")

=== query: what is diabetes? ===

[1] distance=0.2941  meta={'lang': 'en', 'topic': 'diabetes'}
    Diabetes is a chronic metabolic disease characterized by high blood glucose.

[2] distance=1.3449  meta={'lang': 'en', 'topic': 'hypertension'}
    Hypertension is a condition where blood pressure is persistently elevated.

[3] distance=2.0533  meta={'lang': 'zh', 'topic': 'diabetes'}
    糖尿病是一种慢性代谢疾病，特征是血糖升高。



**预期输出**（顺序按距离从小到大）：

```
[1] distance=0.34  meta={'topic': 'diabetes', 'lang': 'en'}
    Diabetes is a chronic metabolic disease characterized by high blood glucose.
[2] distance=0.41  meta={'topic': 'diabetes', 'lang': 'zh'}
    糖尿病是一种慢性代谢疾病，特征是血糖升高。
[3] distance=0.78  meta={'topic': 'hypertension', 'lang': 'en'}
    Hypertension is a condition where blood pressure is persistently elevated.
```

观察重点：

1. **跨语言也能检索到**：英文 query 命中中文文档，这说明 MiniLM 的多语言能力（它实际是英文为主，对中文也能凑合）。如果未来要重度处理中文，应换 `paraphrase-multilingual-MiniLM-L12-v2` 或 `BAAI/bge-m3`。
2. **"无关文档"自然落到最后**：`The dog barked ...` 没出现在 top-3，因为距离更大。
3. **distance 是"距离"不是"相似度"**：越小越相关（这点 LangChain 文档常有歧义，留意）。

---

## 5. Chroma 与"看起来像 Chroma"的几个东西区分

| 名字 | 是什么 | 和 Chroma 的关系 |
|---|---|---|
| **Chroma** | 开源向量数据库（本笔记主角） | — |
| **ChromaDB** | Chroma 的 Python 包名（`pip install chromadb`） | 同一个东西 |
| **`langchain-chroma`** | LangChain 对 Chroma 的封装包 | 包装层，不是底层引擎 |
| **FAISS** | Meta 开源的向量检索库 | Chroma 的同类替代品，更底层、更快、但没集成存储/metadata |
| **LEANN** | 任务文档里提到的备选 | Chroma 的替代品之一 |
| **Pinecone / Weaviate / Qdrant** | 同类向量库 | Chroma 的商业/开源替代品 |
| **Google Chrome** | 浏览器 | 完全无关，只是名字撞了 |

---

## 6. 在你当前 RAG 任务里的角色（联系实战）

下一步「最小 RAG 自验证」（见 `01 验证模型/schedule.md` 末尾章节）里 Chroma 的位置：

```
data/processed/sample.jsonl          ← Day 3 已就绪的 100 篇 PMC 文献
        │
        ▼ (RecursiveCharacterTextSplitter, chunk_size=800)
    ~ 2000 个 chunk
        │
        ▼ (sentence-transformers/all-MiniLM-L6-v2,  384 维)
    2000 个 384 维向量
        │
        ▼  ←  Chroma 在这里登场
   chroma_db/  (PersistentClient)
        │
        │  ←  用户问题 "what is the mechanism of metformin?" 也走相同 embedding
        ▼
   top-4 相关 chunk
        │
        ▼  (拼进 prompt)
   ChatOllama (deepseek-r1:7b)
        │
        ▼
   有据可查的答案
```

**核心收益**：让 deepseek-r1:7b 不再凭"参数里记得的医学知识"作答（Day 2 就答错过二甲双胍机制），而是基于检索到的 PMC 文献片段作答，把幻觉风险压低。

---

## 7. 一句话总结

> Chroma 是**只管"向量怎么存、怎么查"**的数据库。  
> 它**不向量化**——向量化是 embedding 模型干的事。  
> 你给它指定一个 embedding 模型，它就能装作"丢文本进去就能查"，但底层依然是先调模型出向量、再做近邻检索。

记住这两个动作的分工，调参时就不会找错地方：

| 出问题的现象 | 该改哪 |
|---|---|
| 检索结果完全不相关、跨语言效果差 | **换 embedding 模型** |
| 检索很快但召回率低 | 调 chunk_size / chunk_overlap，或换 embedding |
| 检索结果相关但语序乱、想精排 | 在 Chroma 上面接一层 **reranker**（Chroma 自己不做） |
| 集合规模超大、检索变慢 | 调 Chroma 的 HNSW 参数（M / ef_construction）或者换 FAISS / Qdrant |